This notebook analyses top marginal personal income tax rates and macroeconomic outcomes in OECD countries over the approved study period 2000–2022. It includes programmatic data acquisition, data integration and validation, descriptive analysis, two-way fixed-effects regressions, robustness checks, and model diagnostics.


## 0. Environment Setup

In [ ]:
%pip install -q wbgapi linearmodels

In [ ]:
import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats
from linearmodels.panel import PanelOLS
import wbgapi as wb

warnings.filterwarnings("default")


# PROJ518 - Rebuilt Analysis Pipeline

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 7)
plt.rcParams["font.size"] = 11

OECD_COUNTRIES = [
    "AUS","AUT","BEL","CAN","DNK","FIN","FRA","DEU","GRC","IRL",
    "ITA","JPN","KOR","LUX","MEX","NLD","NZL","NOR","PRT","ESP",
    "SWE","CHE","TUR","GBR","USA","ISL","HUN","POL","CZE","SVK"
]

YEAR_START = 2000
YEAR_END = 2022

# Numerical tolerance used when comparing decimal tax-rate changes.
FLOAT_TOL = 1e-9
MIN_TAX_REDUCTION_PP = 0.01
ALT_TAX_REDUCTION_PP = 0.10


def resolve_input_file(preferred_name, glob_pattern):
    """Return an available input file, tolerating duplicate-upload suffixes.

    The preferred exact filename is used first. If it is not present, the
    function looks for matching files such as ``name (1).csv``.
    """
    preferred = Path(preferred_name)
    if preferred.exists():
        return str(preferred)

    candidates = sorted(
        Path(".").glob(glob_pattern),
        key=lambda p: (len(p.name), p.name)
    )

    if not candidates:
        raise FileNotFoundError(
            f"Required input file not found: {preferred_name}. "
            f"Also searched for pattern: {glob_pattern}"
        )

    selected = candidates[0]
    print(f"[INFO] Using input file: {selected.name}")
    return str(selected)


## 1. Data Acquisition

### 1.1 Top Marginal Income Tax Rates

Source: OECD Table I.7, Top statutory personal income tax rates.
The analysis uses the approved study period 2000–2022.


In [ ]:
def load_oecd_tax_rates():
    """
    Load OECD Table I.7 top statutory personal income tax rates for 2000–2022.

    The loader validates the required schema, unit multiplier, study-country
    coverage, year range, numeric fields, and country-year uniqueness.
    """
    OECD_TAX_PATH = resolve_input_file(
        "OECD,DF_TABLE_I7,+all.csv",
        "OECD,DF_TABLE_I7,+all*.csv"
    )

    df = pd.read_csv(OECD_TAX_PATH)
    df.columns = df.columns.str.strip()

    required_cols = {"TAX", "COU", "TIME_PERIOD", "OBS_VALUE"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(
            f"OECD Table I.7 CSV missing expected columns: {missing}. "
            f"Found: {list(df.columns)}"
        )

    df = df[df["TAX"] == "TOP_TRATE"].copy()

    if "UNIT_MULT" in df.columns:
        bad_mult = df.loc[df["UNIT_MULT"].fillna(0) != 0, "UNIT_MULT"]
        if len(bad_mult) > 0:
            raise ValueError(
                f"UNIT_MULT != 0 found ({bad_mult.unique()}). "
                "OBS_VALUE is not in raw percentage-point units."
            )

    df = df.rename(columns={
        "COU": "country_code",
        "TIME_PERIOD": "year",
        "OBS_VALUE": "top_tax_rate"
    })

    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["top_tax_rate"] = pd.to_numeric(df["top_tax_rate"], errors="coerce")

    df = df[["country_code", "year", "top_tax_rate"]].dropna()
    df = df[df["country_code"].isin(OECD_COUNTRIES)]
    df = df[df["year"].between(YEAR_START, YEAR_END)].copy()

    if df.duplicated(["country_code", "year"]).any():
        duplicates = df.loc[
            df.duplicated(["country_code", "year"], keep=False),
            ["country_code", "year"]
        ]
        raise ValueError(
            "Duplicate OECD country-year observations found after filtering:\n"
            f"{duplicates.to_string(index=False)}"
        )

    return df.sort_values(["country_code", "year"]).copy()


In [ ]:
def build_tax_panel():
    """Load and prepare the OECD top statutory personal income tax-rate panel."""
    oecd_tax = load_oecd_tax_rates()

    if oecd_tax.empty:
        raise ValueError("OECD tax-rate data are empty after filtering.")

    return oecd_tax.sort_values(["country_code", "year"]).copy()


print("Loading OECD tax rate data...")
tax_panel = build_tax_panel()
print(
    f"  Tax panel: {len(tax_panel)} obs | "
    f"{tax_panel['year'].min()}-{tax_panel['year'].max()} | "
    f"{tax_panel['country_code'].nunique()} countries"
)


### 1.2 World Bank Indicators
(GDP growth, GFCF, employment, general government final consumption expenditure, inflation, trade, credit)

In [ ]:
WB_INDICATORS = {
    "gdp_growth": "NY.GDP.MKTP.KD.ZG",     # GDP growth (annual %)
    "gfcf_pct_gdp": "NE.GDI.FTOT.ZS",      # Gross fixed capital formation (% GDP)
    "employment": "SL.EMP.TOTL.SP.ZS",     # Employment-to-population ratio (%)
    "gov_spend": "NE.CON.GOVT.ZS",         # General government final consumption expenditure (% GDP)
    "inflation": "FP.CPI.TOTL.ZG",         # Inflation, consumer prices (annual %)
    "trade": "NE.TRD.GNFS.ZS",             # Trade (% GDP)
    "credit": "FS.AST.PRVT.GD.ZS",         # Domestic credit to private sector (% GDP)
}

print("\nFetching World Bank data...")
wb_frames = []
wb_failures = {}

for var_name, indicator_code in WB_INDICATORS.items():
    try:
        df = wb.data.DataFrame(
            indicator_code,
            economy=OECD_COUNTRIES,
            time=range(YEAR_START, YEAR_END + 1),
            labels=False
        )

        # wbgapi returns wide format: rows = countries, columns = years.
        df = df.reset_index()
        df = df.melt(
            id_vars=["economy"],
            var_name="year",
            value_name=var_name
        )
        df = df.rename(columns={"economy": "country_code"})
        df["year"] = (
            df["year"].astype(str).str.extract(r"(\d{4})")[0].astype(int)
        )
        df[var_name] = pd.to_numeric(df[var_name], errors="coerce")

        wb_frames.append(df[["country_code", "year", var_name]])
        print(f"  [{var_name}] OK")

    except Exception as exc:
        wb_failures[var_name] = str(exc)
        print(f"  [{var_name}] FAILED: {exc}")

if wb_failures:
    raise RuntimeError(
        "One or more required World Bank indicators could not be fetched. "
        "Do not continue with an incomplete control-variable set. Failures: "
        f"{wb_failures}"
    )

if len(wb_frames) != len(WB_INDICATORS):
    raise RuntimeError(
        "World Bank data retrieval was incomplete. "
        f"Expected {len(WB_INDICATORS)} indicators, received {len(wb_frames)}."
    )

wb_panel = wb_frames[0]
for frame in wb_frames[1:]:
    wb_panel = wb_panel.merge(
        frame,
        on=["country_code", "year"],
        how="outer",
        validate="one_to_one"
    )

if wb_panel.duplicated(["country_code", "year"]).any():
    raise ValueError("Duplicate country-year observations found in World Bank panel.")

print(f"  WB panel: {len(wb_panel)} obs")


### 1.3 Gini Coefficient

Source: SWIID summary data (`swiid9_92_summary.csv`).
The summary file provides one country-year estimate of the Gini coefficient.
The empirical analysis uses the approved 2000–2022 study period.

The required SWIID CSV file should be stored in the same working directory as this notebook.


In [ ]:
def load_swiid_gini():
    """Load SWIID disposable-income Gini point estimates for study countries."""
    SWIID_PATH = resolve_input_file(
        "swiid9_92_summary.csv",
        "swiid9_92_summary*.csv"
    )

    try:
        swiid = pd.read_csv(SWIID_PATH)

        required_cols = {"country", "year", "gini_disp"}
        missing = required_cols - set(swiid.columns)
        if missing:
            raise ValueError(
                f"SWIID file missing required columns: {missing}. "
                f"Found: {list(swiid.columns)}"
            )

        gini = (
            swiid[["country", "year", "gini_disp"]]
            .rename(columns={
                "country": "country_name",
                "gini_disp": "gini_coefficient"
            })
        )

        name_to_iso = {
            "Australia": "AUS", "Austria": "AUT", "Belgium": "BEL",
            "Canada": "CAN", "Denmark": "DNK", "Finland": "FIN",
            "France": "FRA", "Germany": "DEU", "Greece": "GRC",
            "Ireland": "IRL", "Italy": "ITA", "Japan": "JPN",
            "South Korea": "KOR", "Korea": "KOR", "Luxembourg": "LUX",
            "Mexico": "MEX", "Netherlands": "NLD", "New Zealand": "NZL",
            "Norway": "NOR", "Portugal": "PRT", "Spain": "ESP",
            "Sweden": "SWE", "Switzerland": "CHE", "Turkey": "TUR",
            "United Kingdom": "GBR", "United States": "USA",
            "Iceland": "ISL", "Hungary": "HUN", "Poland": "POL",
            "Czech Republic": "CZE", "Czechia": "CZE", "Slovakia": "SVK",
        }

        gini["country_code"] = gini["country_name"].map(name_to_iso)
        gini["year"] = pd.to_numeric(gini["year"], errors="coerce")
        gini["gini_coefficient"] = pd.to_numeric(
            gini["gini_coefficient"],
            errors="coerce"
        )

        gini = gini.dropna(
            subset=["country_code", "year", "gini_coefficient"]
        )
        gini = gini[gini["country_code"].isin(OECD_COUNTRIES)]
        gini = gini[gini["year"].between(YEAR_START, YEAR_END)].copy()
        gini = gini[
            ["country_code", "year", "gini_coefficient"]
        ].copy()

        if gini.duplicated(["country_code", "year"]).any():
            duplicates = gini.loc[
                gini.duplicated(["country_code", "year"], keep=False),
                ["country_code", "year"]
            ]
            raise ValueError(
                "Duplicate SWIID country-year observations found:\n"
                f"{duplicates.to_string(index=False)}"
            )

        print(
            f"  SWIID Gini: {len(gini)} obs | "
            f"{int(gini['year'].min())}-{int(gini['year'].max())}"
        )
        return gini.sort_values(["country_code", "year"]).copy()

    except Exception as exc:
        raise ValueError(
            f"Unable to load the specified SWIID Gini dataset: {exc}"
        ) from exc


In [ ]:
print("\nLoading Gini data...")
gini_panel = load_swiid_gini()

## 2.1 Build Master Panel

In [ ]:
print("\nMerging all data sources...")

panel = tax_panel.copy()
panel = panel.merge(wb_panel, on=['country_code', 'year'], how='left')


# Filter year range
panel = panel[(panel['year'] >= YEAR_START) & (panel['year'] <= YEAR_END)]

# Drop country-years with no tax rate (core IV)
panel = panel.dropna(subset=['top_tax_rate'])

print(f"\nMaster panel: {len(panel)} obs | "
      f"{panel['country_code'].nunique()} countries | "
      f"{panel['year'].min()}-{panel['year'].max()}")

##2.2 Integration And Validation of Gini Coefficient Data

In [ ]:
# Remove Gini if it already exists
if "gini_coefficient" in panel.columns:
    panel = panel.drop(columns=["gini_coefficient"])

# Verify unique country-year observations
if gini_panel.duplicated(
    ["country_code", "year"]
).any():
    raise ValueError(
        "Duplicate country-year observations found in gini_panel."
    )

# Merge Gini coefficient data into the master panel
panel = panel.merge(
    gini_panel[
        ["country_code", "year", "gini_coefficient"]
    ],
    on=["country_code", "year"],
    how="left",
    validate="one_to_one"
)

# Validate Gini coverage
print("Final master panel shape:", panel.shape)

print(
    "\nGini observations available:",
    panel["gini_coefficient"].notna().sum()
)

print(
    "Gini observations missing:",
    panel["gini_coefficient"].isna().sum()
)

print(
    "Gini coverage:",
    round(
        panel["gini_coefficient"].notna().mean() * 100,
        2
    ),
    "%"
)

print("\nCountries:", panel["country_code"].nunique())

print(
    "Years:",
    panel["year"].min(),
    "to",
    panel["year"].max()
)

## 2.3 Construction of Tax-Reduction Variables

The annual change in the top personal income tax rate is
calculated for each country-year observation. A negative
change represents a reduction in the top personal income
tax rate, while a positive change represents an increase.

Three variables are constructed:


• tax_rate_change — annual change in the top tax rate

• tax_cut — binary indicator for whether a tax reduction occurred

• tax_reduction_pp — magnitude of the tax reduction in
  percentage points

In [ ]:
# Ensure observations are correctly ordered within each country
panel = panel.sort_values(["country_code", "year"]).copy()

# Annual change in the top statutory personal income tax rate.
panel["tax_rate_change"] = (
    panel.groupby("country_code")["top_tax_rate"].diff()
)

# Identify genuine reductions using a small numerical tolerance so tiny
# floating-point artefacts are not treated as policy changes.
panel["tax_cut"] = (
    panel["tax_rate_change"] < -FLOAT_TOL
).astype(int)

# Positive magnitude of a tax-rate reduction, in percentage points.
# Non-reductions remain missing here; the cleaned model measure is created below.
panel["tax_reduction_pp"] = (
    -panel["tax_rate_change"]
).where(panel["tax_rate_change"] < -FLOAT_TOL)

print("Tax-rate change variable created.")

print("\nTax-rate change summary:")
display(panel["tax_rate_change"].describe())

print("\nNumber of tax-reduction observations:")
print(panel["tax_cut"].sum())

print("\nNumber of tax-increase observations:")
print((panel["tax_rate_change"] > FLOAT_TOL).sum())

print("\nNumber of unchanged observations:")
print(
    np.isclose(
        panel["tax_rate_change"],
        0.0,
        atol=FLOAT_TOL,
        rtol=0.0,
        equal_nan=False
    ).sum()
)


## 2.4 Validation of Tax-Reduction Events

The identified tax-reduction observations are examined to assess
their distribution across countries and years and to verify the
magnitude of the recorded changes. This provides a descriptive
validation of the tax-reduction measure before it is incorporated
into the hypothesis-testing models.

In [ ]:
# =================================================
# VALIDATE TAX-REDUCTION EVENTS
# =================================================

tax_cuts = panel.loc[panel["tax_cut"] == 1].copy()

print("TAX-REDUCTION VALIDATION")
print("=" * 60)
print(f"Number of tax-reduction observations: {len(tax_cuts)}")
print(
    "Countries experiencing tax reductions: "
    f"{tax_cuts['country_code'].nunique()}"
)
print(
    "Largest reduction (percentage points): "
    f"{tax_cuts['tax_reduction_pp'].max():.3f}"
)
print(
    "Median reduction (percentage points): "
    f"{tax_cuts['tax_reduction_pp'].median():.3f}"
)


## 2.5 Sensitivity Check for Tax-Reduction Definition

Before proceeding to the main descriptive analysis, the identified
tax reductions are examined for extremely small changes that may
reflect numerical precision rather than economically meaningful
changes in the top personal income tax rate.

In [ ]:
# ============================================================
# SENSITIVITY CHECK: SIZE OF TAX REDUCTIONS
# ============================================================

print("Tax-reduction observations by magnitude:\n")

thresholds = [0.0, 0.01, 0.10, 0.50, 1.00]

for threshold in thresholds:
    if threshold == 0:
        count = (tax_cuts["tax_reduction_pp"] > FLOAT_TOL).sum()
        label = "Reductions greater than 0"
    else:
        count = (
            tax_cuts["tax_reduction_pp"]
            >= (threshold - FLOAT_TOL)
        ).sum()
        label = f"Reductions >= {threshold:g} percentage points"

    print(f"{label}: {count}")

print("\nSmallest tax reductions:")
display(
    tax_cuts[
        [
            "country_code",
            "year",
            "top_tax_rate",
            "tax_rate_change",
            "tax_reduction_pp"
        ]
    ]
    .sort_values("tax_reduction_pp")
    .head(20)
)


## 2.6 Final Tax-Reduction Indicator

Based on the sensitivity analysis, tax reductions smaller than 0.01
percentage points are treated as negligible for the purposes of the
tax-reduction indicator. The original year-on-year tax-rate change
variable is retained, while a cleaned indicator is created for the
subsequent analysis.

In [ ]:
# ============================================================
# FINAL TAX-REDUCTION MEASURES
# ============================================================

# Primary definition: a tax-rate reduction of at least 0.01 percentage points.
# The tolerance prevents values such as 0.009999999999998 from being
# incorrectly excluded when the source value is substantively 0.01.
primary_cut = (
    panel["tax_reduction_pp"].notna()
    & (
        panel["tax_reduction_pp"]
        >= (MIN_TAX_REDUCTION_PP - FLOAT_TOL)
    )
)

panel["tax_cut_clean"] = primary_cut.astype(int)
panel["tax_reduction_clean_pp"] = np.where(
    primary_cut,
    panel["tax_reduction_pp"],
    0.0
)

# Alternative 0.10 percentage-point threshold used for H1 sensitivity analysis.
alternative_cut = (
    panel["tax_reduction_pp"].notna()
    & (
        panel["tax_reduction_pp"]
        >= (ALT_TAX_REDUCTION_PP - FLOAT_TOL)
    )
)

panel["tax_cut_0_10"] = alternative_cut.astype(int)
panel["tax_reduction_0_10_pp"] = np.where(
    alternative_cut,
    panel["tax_reduction_pp"],
    0.0
)

print("Final tax-reduction measures created.")

print("\nPrimary qualifying tax-reduction observations (>= 0.01 pp):")
print(panel["tax_cut_clean"].sum())

print("\nAlternative qualifying observations (>= 0.10 pp):")
print(panel["tax_cut_0_10"].sum())

print("\nNumber of non-tax-reduction observations under primary definition:")
print((panel["tax_cut_clean"] == 0).sum())

print("\nCountries experiencing at least one qualifying tax reduction:")
print(
    panel.loc[
        panel["tax_cut_clean"] == 1,
        "country_code"
    ].nunique()
)

print("\nDistribution of cleaned tax reductions:")
display(
    panel.loc[
        panel["tax_cut_clean"] == 1,
        "tax_reduction_clean_pp"
    ].describe()
)


## 3. Descriptive Analysis

This section provides a descriptive overview of the variables used in
the empirical analysis. The analysis focuses on the top personal income
tax rate, economic growth, income inequality, and the principal
macroeconomic control variables. It also describes the distribution of
tax reductions across countries and over time.

In [ ]:
# ============================================================
# 3.1 Overall Dataset Structure
# ============================================================

print("DESCRIPTIVE OVERVIEW OF THE ANALYTICAL PANEL")
print("=" * 60)

print(f"Number of observations: {len(panel):,}")
print(f"Number of countries: {panel['country_code'].nunique()}")
print(f"Study period: {panel['year'].min()}–{panel['year'].max()}")

print("\nObservations by country:")
print(panel.groupby('country_code').size().describe())

print("\nObservations by year:")
print(panel.groupby('year').size().describe())

In [ ]:
# ============================================================
# 3.2 Descriptive Statistics for Main Variables
# ============================================================

descriptive_vars = [
    'top_tax_rate',
    'gdp_growth',
    'gini_coefficient',
    'gfcf_pct_gdp',
    'employment',
    'gov_spend',
    'inflation',
    'trade',
    'credit'
]

descriptive_stats = panel[descriptive_vars].describe().T

descriptive_stats = descriptive_stats[
    ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']
]

descriptive_stats.columns = [
    'N', 'Mean', 'SD', 'Min', '25th percentile',
    'Median', '75th percentile', 'Max'
]

print("DESCRIPTIVE STATISTICS")
print("=" * 60)

display(descriptive_stats.round(3))

In [ ]:
# ============================================================
# 3.3 Missing Data Assessment
# ============================================================

missing_summary = pd.DataFrame({
    'Missing observations': panel[descriptive_vars].isna().sum(),
    'Missing percentage': (
        panel[descriptive_vars].isna().mean() * 100
    )
})

print("MISSING DATA SUMMARY")
print("=" * 60)

display(missing_summary.round(2))

In [ ]:
# ============================================================
# 3.3.1 Investigation of Missing Credit Observations
# ============================================================

missing_credit = panel[panel['credit'].isna()].copy()

print("MISSING CREDIT OBSERVATIONS")
print("=" * 60)

print(f"Number of missing credit observations: {len(missing_credit)}")

print("\nCountries affected:")
display(
    missing_credit['country_code']
    .value_counts()
    .sort_values(ascending=False)
)

print("\nYears affected:")
display(
    missing_credit['year']
    .value_counts()
    .sort_index()
)

print("\nCountry-year observations with missing credit:")
display(
    missing_credit[['country_code', 'year']]
    .sort_values(['country_code', 'year'])
)

### 3.4 Correlation Analysis

This section examines the pairwise relationships among the principal variables used in the empirical analysis. The correlation analysis provides an initial assessment of the direction and strength of associations between top personal income tax rates, tax reductions, economic growth, income inequality, and the main control variables.

The correlations are descriptive and do not imply causal relationships. They are used primarily to identify broad patterns in the data and to assess whether any variables exhibit potentially strong linear associations that may warrant consideration in the subsequent regression analysis.

In [ ]:
# ============================================================
# 3.4 CORRELATION ANALYSIS
# ============================================================

# Variables used in the empirical analysis
correlation_vars = [
    "top_tax_rate",
    "tax_reduction_clean_pp",
    "gdp_growth",
    "gini_coefficient",
    "gfcf_pct_gdp",
    "employment",
    "gov_spend",
    "inflation",
    "trade",
    "credit"
]

# Check that all required variables are present
missing_corr_vars = [
    var for var in correlation_vars
    if var not in panel.columns
]

if missing_corr_vars:
    print("Missing variables:", missing_corr_vars)
else:
    # Calculate pairwise Pearson correlations
    correlation_matrix = panel[correlation_vars].corr(method="pearson")

    print("CORRELATION MATRIX")
    print("=" * 70)
    display(correlation_matrix.round(3))
    print("\nKEY CORRELATIONS")
    display(
        correlation_matrix.loc[
            ["tax_reduction_clean_pp", "gdp_growth", "gini_coefficient"],
            ["tax_reduction_clean_pp", "gdp_growth", "gini_coefficient"]
        ].round(3)
    )


## 3.5 Tax Reduction Patterns

This section examines the distribution of qualifying reductions in the top personal income tax rate across the analytical panel. It considers the frequency, magnitude, country distribution and timing of tax reductions during the study period. This provides descriptive context for the subsequent econometric analysis.

In [ ]:
# ============================================================
# 3.5.1 SUMMARY OF TAX REDUCTIONS
# ============================================================

# Create qualifying tax-reduction observations for the descriptive summary.
tax_reduction_data = panel.loc[
    panel["tax_reduction_clean_pp"] > 0
].copy()

print("SUMMARY OF TAX REDUCTIONS")
print("=" * 60)

print(f"Number of qualifying tax reductions: {len(tax_reduction_data)}")

print(
    "Countries experiencing at least one qualifying tax reduction: "
    f"{tax_reduction_data['country_code'].nunique()}"
)

print("\nMagnitude of tax reductions:")
display(
    tax_reduction_data["tax_reduction_clean_pp"]
    .describe()
    .round(3)
)


### 3.5.2 Distribution of Tax-Reduction Magnitudes

In [ ]:
# ============================================================
# 3.5.2 DISTRIBUTION OF TAX-REDUCTION MAGNITUDES
# ============================================================

plt.figure(figsize=(8, 5))

plt.hist(
    tax_reduction_data["tax_reduction_clean_pp"],
    bins=15
)

plt.xlabel("Tax reduction (percentage points)")
plt.ylabel("Number of observations")
plt.title("Distribution of Qualifying Tax Reductions")
plt.grid(True, alpha=0.3)

plt.show()


## 3.6 Preliminary Relationships Between Tax Reductions, Economic Growth and Inequality

This section examines the preliminary relationships between qualifying reductions in the top personal income tax rate, economic growth and income inequality. The analysis is descriptive and does not attempt to establish causal effects. The relationships identified here provide a basis for the subsequent fixed-effects regression analysis.

In [ ]:
# ============================================================
# 3.6.1 GDP GROWTH BY TAX-REDUCTION STATUS
# ============================================================

panel["tax_reduction_indicator"] = (
    panel["tax_reduction_clean_pp"] > 0
).astype(int)

panel["tax_reduction_status"] = panel["tax_reduction_indicator"].map({
    0: "No qualifying tax reduction",
    1: "Qualifying tax reduction"
})

growth_by_status = (
    panel.groupby("tax_reduction_status")["gdp_growth"]
    .agg(["count", "mean", "std", "median"])
    .round(3)
)

display(growth_by_status)

In [ ]:
# ============================================================
# 3.6.2 INCOME INEQUALITY BY TAX-REDUCTION STATUS
# ============================================================

print("GINI COEFFICIENT BY TAX-REDUCTION STATUS")
print("=" * 60)

gini_by_status = (
    panel.groupby("tax_reduction_status")["gini_coefficient"]
    .agg(["count", "mean", "std", "median"])
    .round(3)
)

display(gini_by_status)

### 3.6.3 Tax Reductions and GDP Growth

In [ ]:
# ============================================================
# 3.6.3 TAX REDUCTIONS AND GDP GROWTH
# ============================================================


plt.figure(figsize=(8, 5))

plt.scatter(
    panel["tax_reduction_clean_pp"],
    panel["gdp_growth"],
    alpha=0.6
)

plt.xlabel("Tax reduction (percentage points)")
plt.ylabel("GDP growth (%)")
plt.title("Tax Reductions and GDP Growth")

plt.grid(True, alpha=0.3)
plt.show()

### 3.6.4 Tax Reductions and Income Inequality

In [ ]:
# ============================================================
# 3.6.4 TAX REDUCTIONS AND INCOME INEQUALITY
# ============================================================

plt.figure(figsize=(8, 5))

plt.scatter(
    panel["tax_reduction_clean_pp"],
    panel["gini_coefficient"],
    alpha=0.6
)

plt.xlabel("Tax reduction (percentage points)")
plt.ylabel("Gini coefficient")
plt.title("Tax Reductions and Income Inequality")

plt.grid(True, alpha=0.3)
plt.show()

## 3.7 Fixed-Effects Regression Analysis

The preliminary analysis identified only weak unconditional relationships between tax reductions, economic growth and income inequality. To account for unobserved characteristics that may be specific to individual countries and common shocks affecting countries over time, the analysis now employs two-way fixed-effects regression models with country and year fixed effects.

The models also control for relevant macroeconomic and investment variables included in the analytical panel. Standard errors are clustered at the country level to account for within-country dependence over time.

In [ ]:
# ============================================================
# 3.7.0 H1 REGRESSION SAMPLE
# ============================================================

h1_variables = [
    "gdp_growth",
    "tax_reduction_clean_pp",
    "tax_reduction_0_10_pp",
    "gfcf_pct_gdp",
    "employment",
    "gov_spend",
    "inflation",
    "trade",
    "credit",
    "country_code",
    "year"
]

h1_sample = panel[h1_variables].dropna().copy()

print("H1 REGRESSION SAMPLE")
print("=" * 60)
print(f"Observations: {len(h1_sample)}")
print(f"Countries: {h1_sample['country_code'].nunique()}")
print(f"Years: {h1_sample['year'].nunique()}")

print("\nMissing values in H1 sample:")
display(h1_sample.isna().sum())


### 3.7.1 H1: Tax Reductions and Economic Growth

A two-way fixed-effects regression is estimated to examine whether qualifying reductions in the top personal income tax rate are associated with GDP growth. The model includes country and year fixed effects and controls for gross fixed capital formation, employment, government expenditure, inflation, trade openness, and credit. Standard errors are clustered at the country level.

In [ ]:
# ============================================================
# 3.7.1 H1: TWO-WAY FIXED-EFFECTS MODEL
# Tax Reductions and GDP Growth
# ============================================================

import statsmodels.formula.api as smf

h1_model = smf.ols(
    """
    gdp_growth ~ tax_reduction_clean_pp
    + gfcf_pct_gdp
    + employment
    + gov_spend
    + inflation
    + trade
    + credit
    + C(country_code)
    + C(year)
    """,
    data=h1_sample
).fit(
    cov_type="cluster",
    cov_kwds={"groups": h1_sample["country_code"]}
)

print("H1: TAX REDUCTIONS AND GDP GROWTH")
print("=" * 60)
print(h1_model.summary())

### 3.7.2 H1 Model Diagnostic: Multicollinearity Check

Before interpreting the H1 regression as the main inferential result, the specification is assessed for potential multicollinearity among the continuous explanatory variables. This check is used to determine whether the large condition number reported by the regression model reflects problematic correlation among the substantive predictors.

In [ ]:
# ============================================================
# 3.7.2 H1 MODEL DIAGNOSTIC: POOLED VIF
# ============================================================

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

vif_variables = h1_sample[
    [
        "tax_reduction_clean_pp",
        "gfcf_pct_gdp",
        "employment",
        "gov_spend",
        "inflation",
        "trade",
        "credit"
    ]
].copy()

# Conventional VIF requires an intercept in the auxiliary regressions.
vif_with_constant = add_constant(vif_variables, has_constant="add")

vif_results = pd.DataFrame({
    "variable": vif_variables.columns,
    "VIF": [
        variance_inflation_factor(
            vif_with_constant.values,
            i
        )
        for i in range(1, vif_with_constant.shape[1])
    ]
})

vif_results["VIF"] = vif_results["VIF"].round(3)

print("Pooled VIFs (with intercept in auxiliary regressions):")
display(vif_results)


### 3.7.3 Fixed-Effects-Adjusted Multicollinearity Diagnostic

Because the H1 specification includes country and year fixed effects, multicollinearity is assessed after accounting for these fixed effects. This provides a more appropriate diagnostic of the linear dependence among the substantive explanatory variables used to estimate the within-country relationships.

In [ ]:
# ============================================================
# 3.7.3 FIXED-EFFECTS-ADJUSTED VIF
# ============================================================

from statsmodels.stats.outliers_influence import variance_inflation_factor


def residualize_two_way_fe(
    data,
    variables,
    entity_col="country_code",
    time_col="year"
):
    """Residualise variables on entity and time fixed effects.

    Using OLS residualisation is valid for balanced and unbalanced panels,
    unlike simple double-demeaning in an unbalanced sample.
    """
    adjusted = pd.DataFrame(index=data.index)

    for variable in variables:
        fe_model = smf.ols(
            f"{variable} ~ C({entity_col}) + C({time_col})",
            data=data
        ).fit()

        adjusted[variable] = fe_model.resid.reindex(data.index)

    return adjusted


def calculate_vif(adjusted_data, label="FE-adjusted VIF"):
    """Calculate VIFs for a complete residualised design matrix."""
    complete = adjusted_data.dropna().copy()

    return pd.DataFrame({
        "variable": complete.columns,
        label: [
            variance_inflation_factor(complete.values, i)
            for i in range(complete.shape[1])
        ]
    })


fe_vif_vars = [
    "tax_reduction_clean_pp",
    "gfcf_pct_gdp",
    "employment",
    "gov_spend",
    "inflation",
    "trade",
    "credit"
]

h1_fe_adjusted = residualize_two_way_fe(
    h1_sample,
    fe_vif_vars
)

fe_vif_results = calculate_vif(
    h1_fe_adjusted,
    label="FE-adjusted VIF"
)

fe_vif_results["FE-adjusted VIF"] = (
    fe_vif_results["FE-adjusted VIF"].round(3)
)

display(fe_vif_results)


### 3.7.4 H1 Robustness Checks

The primary continuous tax-reduction measure is supplemented with (1) a binary qualifying tax-reduction indicator and (2) an alternative 0.10 percentage-point minimum threshold. These checks assess whether the H1 result depends on the exact treatment definition.

In [ ]:
# ============================================================
# 3.7.4 H1 ROBUSTNESS CHECKS
# ============================================================

print("H1 ROBUSTNESS CHECKS")
print("=" * 70)

# 1) Binary indicator under the primary >= 0.01 pp definition
h1_sample["tax_reduction_indicator"] = (
    h1_sample["tax_reduction_clean_pp"] > 0
).astype(int)

h1_binary_model = smf.ols(
    """
    gdp_growth ~ tax_reduction_indicator
    + gfcf_pct_gdp
    + employment
    + gov_spend
    + inflation
    + trade
    + credit
    + C(country_code)
    + C(year)
    """,
    data=h1_sample
).fit(
    cov_type="cluster",
    cov_kwds={"groups": h1_sample["country_code"]}
)

# 2) Continuous magnitude using a stricter >= 0.10 pp threshold
h1_threshold_model = smf.ols(
    """
    gdp_growth ~ tax_reduction_0_10_pp
    + gfcf_pct_gdp
    + employment
    + gov_spend
    + inflation
    + trade
    + credit
    + C(country_code)
    + C(year)
    """,
    data=h1_sample
).fit(
    cov_type="cluster",
    cov_kwds={"groups": h1_sample["country_code"]}
)

h1_robustness_summary = pd.DataFrame({
    "Specification": [
        "Primary continuous measure (>= 0.01 pp)",
        "Binary tax-reduction indicator",
        "Continuous measure with >= 0.10 pp threshold"
    ],
    "Key coefficient": [
        h1_model.params["tax_reduction_clean_pp"],
        h1_binary_model.params["tax_reduction_indicator"],
        h1_threshold_model.params["tax_reduction_0_10_pp"]
    ],
    "SE": [
        h1_model.bse["tax_reduction_clean_pp"],
        h1_binary_model.bse["tax_reduction_indicator"],
        h1_threshold_model.bse["tax_reduction_0_10_pp"]
    ],
    "p-value": [
        h1_model.pvalues["tax_reduction_clean_pp"],
        h1_binary_model.pvalues["tax_reduction_indicator"],
        h1_threshold_model.pvalues["tax_reduction_0_10_pp"]
    ]
})

h1_robustness_summary[["Key coefficient", "SE", "p-value"]] = (
    h1_robustness_summary[
        ["Key coefficient", "SE", "p-value"]
    ].round(4)
)

display(h1_robustness_summary)


### 3.8.1 H2 Regression Sample

The H2 regression examines the association between qualifying reductions in the top personal income tax rate and income inequality, measured by the Gini coefficient. The analysis uses the complete-case sample for the dependent variable, the tax-reduction measure, and all specified control variables. Country and year fixed effects are retained, with standard errors clustered at the country level.

In [ ]:
# ============================================================
# 3.8.1 H2 REGRESSION SAMPLE
# ============================================================

h2_variables = [
    "gini_coefficient",
    "tax_reduction_clean_pp",
    "gfcf_pct_gdp",
    "employment",
    "gov_spend",
    "inflation",
    "trade",
    "credit",
    "country_code",
    "year"
]

h2_sample = (
    panel[h2_variables]
    .dropna()
    .copy()
)

print("H2 REGRESSION SAMPLE")
print("=" * 60)
print(f"Observations: {len(h2_sample)}")
print(f"Countries: {h2_sample['country_code'].nunique()}")
print(f"Years: {h2_sample['year'].nunique()}")

print("\nMissing values in H2 sample:")
display(h2_sample.isna().sum())

In [ ]:
# ============================================================
# 3.8.2 H2: TAX REDUCTIONS AND INCOME INEQUALITY
# ============================================================

print("H2: TAX REDUCTIONS AND INCOME INEQUALITY")
print("=" * 70)

h2_model = smf.ols(
    """
    gini_coefficient
    ~ tax_reduction_clean_pp
    + gfcf_pct_gdp
    + employment
    + gov_spend
    + inflation
    + trade
    + credit
    + C(country_code)
    + C(year)
    """,
    data=h2_sample
).fit(
    cov_type="cluster",
    cov_kwds={"groups": h2_sample["country_code"]}
)

print(h2_model.summary())

### 3.8.3 H2 Model Diagnostic: VIF

In [ ]:
# ============================================================
# 3.8.3 H2 MODEL DIAGNOSTIC: FE-ADJUSTED VIF
# ============================================================

h2_vif_vars = [
    "tax_reduction_clean_pp",
    "gfcf_pct_gdp",
    "employment",
    "gov_spend",
    "inflation",
    "trade",
    "credit"
]

# OLS residualisation correctly removes country and year effects even though
# H2 is an unbalanced panel after complete-case deletion.
h2_vif_data = residualize_two_way_fe(
    h2_sample,
    h2_vif_vars
)

h2_fe_vif_results = calculate_vif(
    h2_vif_data,
    label="FE-adjusted VIF"
)

h2_fe_vif_results["FE-adjusted VIF"] = (
    h2_fe_vif_results["FE-adjusted VIF"].round(3)
)

display(h2_fe_vif_results)


### 3.9.1 H3 Regression Sample

In [ ]:
# ============================================================
# 3.9.1 H3 REGRESSION SAMPLE
# ============================================================

h3_variables = [
    "gdp_growth",
    "tax_reduction_clean_pp",
    "gini_coefficient",
    "gfcf_pct_gdp",
    "employment",
    "gov_spend",
    "inflation",
    "trade",
    "credit",
    "country_code",
    "year"
]

h3_sample = (
    panel[h3_variables]
    .dropna()
    .copy()
)

print("H3 REGRESSION SAMPLE")
print("=" * 60)
print(f"Observations: {len(h3_sample)}")
print(f"Countries: {h3_sample['country_code'].nunique()}")
print(f"Years: {h3_sample['year'].nunique()}")

print("\nMissing values in H3 sample:")
display(h3_sample.isna().sum())

### 3.9.2 H3 Interaction Term

In [ ]:
# ============================================================
# 3.9.2 H3 INTERACTION TERM
# ============================================================

h3_sample["tax_gini_interaction"] = (
    h3_sample["tax_reduction_clean_pp"]
    * h3_sample["gini_coefficient"]
)

print("H3 INTERACTION TERM")
print("=" * 60)

print("Interaction variable created:")
print("tax_reduction_clean_pp × gini_coefficient")
print()

print("Descriptive statistics:")
display(
    h3_sample[
        [
            "tax_reduction_clean_pp",
            "gini_coefficient",
            "tax_gini_interaction"
        ]
    ].describe()
)

### 3.9.3 H3: Tax Reductions × Inequality and Economic Growth

In [ ]:
# ============================================================
# 3.9.3 H3: TAX REDUCTIONS × INEQUALITY AND ECONOMIC GROWTH
# ============================================================

print("H3: TAX REDUCTIONS × INEQUALITY AND ECONOMIC GROWTH")
print("=" * 70)

h3_model = smf.ols(
    """
    gdp_growth
    ~ tax_reduction_clean_pp
    + gini_coefficient
    + tax_gini_interaction
    + gfcf_pct_gdp
    + employment
    + gov_spend
    + inflation
    + trade
    + credit
    + C(country_code)
    + C(year)
    """,
    data=h3_sample
).fit(
    cov_type="cluster",
    cov_kwds={"groups": h3_sample["country_code"]}
)

print(h3_model.summary())

### 3.9.4 Marginal Effects and Statistical Significance of Tax Reductions Across Inequality


In [ ]:
# ============================================================
# 3.9.4 MARGINAL EFFECTS AND STATISTICAL SIGNIFICANCE OF TAX
# REDUCTIONS ACROSS INEQUALITY
# ============================================================

# Extract coefficients and the cluster-robust covariance matrix
beta_tax = h3_model.params["tax_reduction_clean_pp"]
beta_interaction = h3_model.params["tax_gini_interaction"]
cov = h3_model.cov_params()

# Representative inequality levels
# (25th percentile, mean, and 75th percentile of the H3 sample)
gini_levels = {
    "Low inequality (25th percentile)": h3_sample["gini_coefficient"].quantile(0.25),
    "Mean inequality": h3_sample["gini_coefficient"].mean(),
    "High inequality (75th percentile)": h3_sample["gini_coefficient"].quantile(0.75)
}

# Variance and covariance components used for the delta-method SE
var_tax = cov.loc[
    "tax_reduction_clean_pp",
    "tax_reduction_clean_pp"
]

var_interaction = cov.loc[
    "tax_gini_interaction",
    "tax_gini_interaction"
]

cov_tax_interaction = cov.loc[
    "tax_reduction_clean_pp",
    "tax_gini_interaction"
]

results = []

for label, gini_value in gini_levels.items():

    # Conditional/marginal effect of tax reduction at the specified Gini level
    marginal_effect = (
        beta_tax
        + beta_interaction * gini_value
    )

    # Delta-method variance for the conditional effect
    marginal_variance = (
        var_tax
        + (gini_value ** 2) * var_interaction
        + 2 * gini_value * cov_tax_interaction
    )

    marginal_se = np.sqrt(marginal_variance)
    z_value = marginal_effect / marginal_se
    p_value = 2 * (1 - stats.norm.cdf(abs(z_value)))

    ci_lower = marginal_effect - 1.96 * marginal_se
    ci_upper = marginal_effect + 1.96 * marginal_se

    results.append([
        label,
        gini_value,
        marginal_effect,
        marginal_se,
        z_value,
        p_value,
        ci_lower,
        ci_upper
    ])

marginal_effect_results = pd.DataFrame(
    results,
    columns=[
        "Inequality level",
        "Gini",
        "Marginal effect",
        "SE",
        "z",
        "p-value",
        "95% CI lower",
        "95% CI upper"
    ]
)

print("MARGINAL EFFECT OF TAX REDUCTIONS AT DIFFERENT GINI LEVELS")
print("=" * 70)

display(
    marginal_effect_results.round(4)
)


### 3.9.6 H3 Model Diagnostic: FE-Adjusted VIF

In [ ]:
# ============================================================
# 3.9.6 H3 MODEL DIAGNOSTIC: FE-ADJUSTED VIF
# ============================================================

h3_vif_vars = [
    "tax_reduction_clean_pp",
    "gini_coefficient",
    "tax_gini_interaction",
    "gfcf_pct_gdp",
    "employment",
    "gov_spend",
    "inflation",
    "trade",
    "credit"
]

h3_vif_data = residualize_two_way_fe(
    h3_sample,
    h3_vif_vars
)

h3_fe_vif_results = calculate_vif(
    h3_vif_data,
    label="FE-adjusted VIF"
)

h3_fe_vif_results["FE-adjusted VIF"] = (
    h3_fe_vif_results["FE-adjusted VIF"].round(3)
)

display(h3_fe_vif_results)


### 3.9.7 Centred H3 Interaction Term

In [ ]:
# ============================================================
# 3.9.7 CENTRED H3 INTERACTION TERM
# ============================================================

# Centre Gini around its sample mean
h3_sample["gini_centered"] = (
    h3_sample["gini_coefficient"]
    - h3_sample["gini_coefficient"].mean()
)

# Construct centred interaction
h3_sample["tax_gini_interaction_centered"] = (
    h3_sample["tax_reduction_clean_pp"]
    * h3_sample["gini_centered"]
)

print("CENTRED H3 INTERACTION TERM")
print("=" * 70)

print(
    f"Mean Gini: "
    f"{h3_sample['gini_coefficient'].mean():.3f}"
)

print(
    f"Mean centred Gini: "
    f"{h3_sample['gini_centered'].mean():.6f}"
)

print("\nDescriptive statistics:")
display(
    h3_sample[
        [
            "tax_reduction_clean_pp",
            "gini_coefficient",
            "gini_centered",
            "tax_gini_interaction_centered"
        ]
    ].describe()
)

### 3.9.8 H3: CENTRED TAX REDUCTION × INEQUALITY MODEL

In [ ]:
# ============================================================
# 3.9.8 H3: CENTRED TAX REDUCTION × INEQUALITY MODEL
# ============================================================

print("H3: CENTRED TAX REDUCTION × INEQUALITY MODEL")
print("=" * 70)

h3_centered_model = smf.ols(
    """
    gdp_growth
    ~ tax_reduction_clean_pp
    + gini_centered
    + tax_gini_interaction_centered
    + gfcf_pct_gdp
    + employment
    + gov_spend
    + inflation
    + trade
    + credit
    + C(country_code)
    + C(year)
    """,
    data=h3_sample
).fit(
    cov_type="cluster",
    cov_kwds={"groups": h3_sample["country_code"]}
)

print(h3_centered_model.summary())

### 3.9.9 H3 Centred Model Diagnostic: FE-Adjusted VIF

In [ ]:
# ============================================================
# 3.9.9 H3 CENTRED MODEL DIAGNOSTIC: FE-ADJUSTED VIF
# ============================================================

h3_centered_vif_vars = [
    "tax_reduction_clean_pp",
    "gini_centered",
    "tax_gini_interaction_centered",
    "gfcf_pct_gdp",
    "employment",
    "gov_spend",
    "inflation",
    "trade",
    "credit"
]

h3_centered_vif_data = residualize_two_way_fe(
    h3_sample,
    h3_centered_vif_vars
)

h3_centered_fe_vif_results = calculate_vif(
    h3_centered_vif_data,
    label="FE-adjusted VIF"
)

h3_centered_fe_vif_results["FE-adjusted VIF"] = (
    h3_centered_fe_vif_results["FE-adjusted VIF"].round(3)
)

display(h3_centered_fe_vif_results)


### 3.9.10 H3 Interaction Plot

In [ ]:
# ============================================================
# 3.9.10 H3 INTERACTION PLOT
# Average adjusted predictions from the full fitted model
# ============================================================

gini_plot_levels = {
    "Low inequality (25th percentile)": h3_sample["gini_coefficient"].quantile(0.25),
    "Mean inequality": h3_sample["gini_coefficient"].mean(),
    "High inequality (75th percentile)": h3_sample["gini_coefficient"].quantile(0.75)
}

tax_range = np.linspace(
    h3_sample["tax_reduction_clean_pp"].min(),
    h3_sample["tax_reduction_clean_pp"].max(),
    100
)

mean_gini_for_plot = h3_sample["gini_coefficient"].mean()
base_prediction_data = h3_sample.copy()

plt.figure(figsize=(9, 6))

for label, gini_value in gini_plot_levels.items():

    gini_centered_value = gini_value - mean_gini_for_plot
    predicted_curve = []

    for tax_value in tax_range:
        prediction_data = base_prediction_data.copy()

        # Set the focal predictors while retaining each observation's actual
        # controls, country effect and year effect.
        prediction_data["tax_reduction_clean_pp"] = tax_value
        prediction_data["gini_centered"] = gini_centered_value
        prediction_data["tax_gini_interaction_centered"] = (
            tax_value * gini_centered_value
        )

        # Average full-model predictions over the observed covariate,
        # country and year distribution.
        predicted_curve.append(
            h3_centered_model.predict(prediction_data).mean()
        )

    plt.plot(
        tax_range,
        predicted_curve,
        label=f"{label} (Gini = {gini_value:.2f})"
    )

plt.xlabel("Tax reduction (percentage points)")
plt.ylabel("Average adjusted predicted GDP growth (%)")
plt.title(
    "H3: Tax Reductions, Income Inequality and Economic Growth"
)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


### 3.9.11 H3 Robustness Check: Tax-Reduction Indicator × Inequality

In [ ]:
# ============================================================
# 3.9.11 H3 ROBUSTNESS: INDICATOR × INEQUALITY
# ============================================================

# Create binary tax-reduction indicator
h3_sample["tax_reduction_indicator"] = (
    h3_sample["tax_reduction_clean_pp"] > 0
).astype(int)

# Create interaction with centred Gini
h3_sample["indicator_gini_interaction"] = (
    h3_sample["tax_reduction_indicator"]
    * h3_sample["gini_centered"]
)

print("H3 ROBUSTNESS: TAX-REDUCTION INDICATOR × INEQUALITY")
print("=" * 70)

print(
    f"Qualifying tax-reduction observations: "
    f"{h3_sample['tax_reduction_indicator'].sum()}"
)

print(
    f"No qualifying tax-reduction observations: "
    f"{(h3_sample['tax_reduction_indicator'] == 0).sum()}"
)

print("\nInteraction descriptive statistics:")

display(
    h3_sample[
        [
            "tax_reduction_indicator",
            "gini_centered",
            "indicator_gini_interaction"
        ]
    ].describe()
)

### 3.9.12 H3 Indicator Interaction Model

In [ ]:
# ============================================================
# 3.9.12 H3 INDICATOR INTERACTION MODEL
# ============================================================

print("H3 ROBUSTNESS: TAX-REDUCTION INDICATOR × INEQUALITY")
print("=" * 70)

h3_indicator_model = smf.ols(
    """
    gdp_growth
    ~ tax_reduction_indicator
    + gini_centered
    + indicator_gini_interaction
    + gfcf_pct_gdp
    + employment
    + gov_spend
    + inflation
    + trade
    + credit
    + C(country_code)
    + C(year)
    """,
    data=h3_sample
).fit(
    cov_type="cluster",
    cov_kwds={"groups": h3_sample["country_code"]}
)

print(h3_indicator_model.summary())

In [ ]:
# ============================================================
# 3.9.13 H3 ROBUSTNESS MODEL: FE-ADJUSTED VIF
# ============================================================

print("H3 ROBUSTNESS: FE-ADJUSTED VIF")
print("=" * 70)

h3_indicator_vif_vars = [
    "tax_reduction_indicator",
    "gini_centered",
    "indicator_gini_interaction",
    "gfcf_pct_gdp",
    "employment",
    "gov_spend",
    "inflation",
    "trade",
    "credit"
]

vif_adjusted_h3_indicator = residualize_two_way_fe(
    h3_sample,
    h3_indicator_vif_vars
)

vif_results_h3_indicator = calculate_vif(
    vif_adjusted_h3_indicator,
    label="FE-adjusted VIF"
)

vif_results_h3_indicator["FE-adjusted VIF"] = (
    vif_results_h3_indicator["FE-adjusted VIF"].round(3)
)

display(vif_results_h3_indicator)


### 3.9.14 H3 Robustness: Marginal Effects of Tax-Reduction Indicator at Different Levels of Inequality

In [ ]:
# ============================================================
# 3.9.14 H3 ROBUSTNESS: MARGINAL EFFECTS OF TAX-REDUCTION
# INDICATOR AT DIFFERENT LEVELS OF INEQUALITY
# ============================================================

print("H3 ROBUSTNESS: MARGINAL EFFECTS AT DIFFERENT GINI LEVELS")
print("=" * 70)

# Gini values from the H3 sample
gini_low = h3_sample["gini_coefficient"].quantile(0.25)
gini_mean = h3_sample["gini_coefficient"].mean()
gini_high = h3_sample["gini_coefficient"].quantile(0.75)

# Mean Gini used for centering
mean_gini = h3_sample["gini_coefficient"].mean()

gini_levels = {
    "Low inequality (25th percentile)": gini_low,
    "Mean inequality": gini_mean,
    "High inequality (75th percentile)": gini_high
}

# Use the correctly named fitted model
beta_indicator = h3_indicator_model.params[
    "tax_reduction_indicator"
]

beta_interaction = h3_indicator_model.params[
    "indicator_gini_interaction"
]

cov_matrix = h3_indicator_model.cov_params()

var_indicator = cov_matrix.loc[
    "tax_reduction_indicator",
    "tax_reduction_indicator"
]

var_interaction = cov_matrix.loc[
    "indicator_gini_interaction",
    "indicator_gini_interaction"
]

cov_indicator_interaction = cov_matrix.loc[
    "tax_reduction_indicator",
    "indicator_gini_interaction"
]

marginal_effect_results = []

for label, gini_value in gini_levels.items():

    gini_centered_value = gini_value - mean_gini

    marginal_effect = (
        beta_indicator
        + beta_interaction * gini_centered_value
    )

    marginal_variance = (
        var_indicator
        + (gini_centered_value ** 2) * var_interaction
        + 2 * gini_centered_value * cov_indicator_interaction
    )

    marginal_se = np.sqrt(marginal_variance)

    z_value = marginal_effect / marginal_se

    p_value = 2 * (
        1 - stats.norm.cdf(abs(z_value))
    )

    ci_lower = marginal_effect - 1.96 * marginal_se
    ci_upper = marginal_effect + 1.96 * marginal_se

    marginal_effect_results.append({
        "Inequality level": label,
        "Gini": gini_value,
        "Marginal effect": marginal_effect,
        "SE": marginal_se,
        "z": z_value,
        "p-value": p_value,
        "95% CI lower": ci_lower,
        "95% CI upper": ci_upper
    })

h3_indicator_marginal_effects = pd.DataFrame(
    marginal_effect_results
)

numeric_columns = [
    "Gini",
    "Marginal effect",
    "SE",
    "z",
    "p-value",
    "95% CI lower",
    "95% CI upper"
]

h3_indicator_marginal_effects[numeric_columns] = (
    h3_indicator_marginal_effects[numeric_columns].round(4)
)

display(h3_indicator_marginal_effects)

### 3.10 Model Diagnostics

### 3.10.1 H1–H3 Residual Heteroskedasticity Checks

In [ ]:
# ============================================================
# 3.10.1 H1–H3 RESIDUAL HETEROSKEDASTICITY CHECKS
# ============================================================

from statsmodels.stats.diagnostic import het_breuschpagan

models_for_diagnostics = {
    "H1": h1_model,
    "H2": h2_model,
    "H3": h3_centered_model
}

print("HETEROSKEDASTICITY DIAGNOSTICS")
print("=" * 70)

heteroskedasticity_results = []

for name, model in models_for_diagnostics.items():

    # Breusch-Pagan test
    bp_test = het_breuschpagan(
        model.resid,
        model.model.exog
    )

    lm_stat = bp_test[0]
    lm_pvalue = bp_test[1]
    f_stat = bp_test[2]
    f_pvalue = bp_test[3]

    heteroskedasticity_results.append({
        "Model": name,
        "LM statistic": lm_stat,
        "LM p-value": lm_pvalue,
        "F statistic": f_stat,
        "F p-value": f_pvalue
    })

heteroskedasticity_results = pd.DataFrame(
    heteroskedasticity_results
)

heteroskedasticity_results[
    [
        "LM statistic",
        "LM p-value",
        "F statistic",
        "F p-value"
    ]
] = heteroskedasticity_results[
    [
        "LM statistic",
        "LM p-value",
        "F statistic",
        "F p-value"
    ]
].round(4)

display(heteroskedasticity_results)

### 3.10.2 Panel Serial-Correlation Diagnostics


In [ ]:
# ============================================================
# 3.10.2 PANEL SERIAL-CORRELATION DIAGNOSTICS
# Durbin-Watson and consecutive-year within-country residual AR(1)
# ============================================================

from statsmodels.stats.stattools import durbin_watson

print("PANEL SERIAL-CORRELATION DIAGNOSTICS")
print("=" * 70)

serial_models = {
    "H1": (h1_model, h1_sample),
    "H2": (h2_model, h2_sample),
    "H3": (h3_centered_model, h3_sample)
}


def residual_serial_correlation(model, data):
    """Correlation of residuals with the immediately preceding calendar year."""
    row_labels = list(model.model.data.row_labels)

    diagnostic_df = data.loc[
        row_labels,
        ["country_code", "year"]
    ].copy()

    diagnostic_df["residual"] = np.asarray(model.resid)
    diagnostic_df = diagnostic_df.sort_values(
        ["country_code", "year"]
    )

    diagnostic_df["lag_year"] = (
        diagnostic_df
        .groupby("country_code")["year"]
        .shift(1)
    )

    diagnostic_df["lag_residual"] = (
        diagnostic_df
        .groupby("country_code")["residual"]
        .shift(1)
    )

    # Retain only genuine t and t-1 pairs. This avoids treating, for example,
    # 2008 and 2010 as adjacent when 2009 is missing.
    test_df = diagnostic_df.loc[
        (diagnostic_df["year"] - diagnostic_df["lag_year"]) == 1
    ].dropna(subset=["residual", "lag_residual"]).copy()

    if len(test_df) < 2:
        return np.nan, len(test_df)

    correlation = test_df[
        ["residual", "lag_residual"]
    ].corr().iloc[0, 1]

    return correlation, len(test_df)


serial_results = []

for name, (model, data) in serial_models.items():
    dw = durbin_watson(model.resid)
    ar1, ar1_n = residual_serial_correlation(model, data)

    serial_results.append({
        "Model": name,
        "Durbin-Watson (supplementary)": round(dw, 4),
        "Consecutive-year residual AR(1)": round(ar1, 4),
        "Consecutive-year pairs": ar1_n
    })

serial_results = pd.DataFrame(serial_results)

display(serial_results)


### 3.11.1 H2 Panel Fixed-Effects Robustness

In [ ]:
# ============================================================
# 3.11.1 H2 PANEL FIXED-EFFECTS ROBUSTNESS
# ============================================================

from linearmodels.panel import PanelOLS

print("H2 PANEL FIXED-EFFECTS ROBUSTNESS")
print("=" * 70)

# Prepare H2 panel data
h2_panel = h2_sample.copy()

h2_panel = h2_panel.set_index(
    ["country_code", "year"]
)

# Define dependent variable
y_h2 = h2_panel["gini_coefficient"]

# Define explanatory variables
X_h2 = h2_panel[
    [
        "tax_reduction_clean_pp",
        "gfcf_pct_gdp",
        "employment",
        "gov_spend",
        "inflation",
        "trade",
        "credit"
    ]
]

# Estimate country fixed-effects model
h2_panel_model = PanelOLS(
    y_h2,
    X_h2,
    entity_effects=True,
    time_effects=True
).fit(
    cov_type="clustered",
    cluster_entity=True
)

print(h2_panel_model)

### 3.11.2 H2 Robustness: Tax-Reduction Indicator

In [ ]:
# ============================================================
# 3.11.2 H2 ROBUSTNESS: TAX-REDUCTION INDICATOR
# ============================================================

print("H2 ROBUSTNESS: TAX-REDUCTION INDICATOR")
print("=" * 70)

# Create binary tax-reduction indicator if it does not already exist
h2_sample["tax_reduction_indicator"] = (
    h2_sample["tax_reduction_clean_pp"] > 0
).astype(int)

h2_indicator_model = smf.ols(
    """
    gini_coefficient
    ~ tax_reduction_indicator
    + gfcf_pct_gdp
    + employment
    + gov_spend
    + inflation
    + trade
    + credit
    + C(country_code)
    + C(year)
    """,
    data=h2_sample
).fit(
    cov_type="cluster",
    cov_kwds={"groups": h2_sample["country_code"]}
)

print(h2_indicator_model.summary())

### 3.11.3 H2 Robustness Summary

In [ ]:
# ============================================================
# 3.11.3 H2 ROBUSTNESS SUMMARY
# ============================================================

print("H2 ROBUSTNESS SUMMARY")
print("=" * 70)

h2_robustness_summary = pd.DataFrame({
    "Specification": [
        "Primary continuous measure",
        "Binary tax-reduction indicator",
        "PanelOLS two-way fixed effects"
    ],
    "Tax-reduction coefficient": [
        h2_model.params["tax_reduction_clean_pp"],
        h2_indicator_model.params["tax_reduction_indicator"],
        h2_panel_model.params["tax_reduction_clean_pp"]
    ],
    "p-value": [
        h2_model.pvalues["tax_reduction_clean_pp"],
        h2_indicator_model.pvalues["tax_reduction_indicator"],
        h2_panel_model.pvalues["tax_reduction_clean_pp"]
    ]
})

h2_robustness_summary[
    ["Tax-reduction coefficient", "p-value"]
] = h2_robustness_summary[
    ["Tax-reduction coefficient", "p-value"]
].round(4)

display(h2_robustness_summary)

### 3.12 Final Sample and Model Specification Audit

In [ ]:
# ============================================================
# 3.12 FINAL SAMPLE AND MODEL SPECIFICATION AUDIT
# ============================================================

print("FINAL SAMPLE CONSISTENCY CHECK")
print("=" * 70)

h1_n = int(h1_model.nobs)
h2_n = int(h2_model.nobs)
h3_n = int(h3_centered_model.nobs)

h1_start = int(h1_sample["year"].min())
h1_end = int(h1_sample["year"].max())

h2_start = int(h2_sample["year"].min())
h2_end = int(h2_sample["year"].max())

h3_start = int(h3_sample["year"].min())
h3_end = int(h3_sample["year"].max())

print(f"H1 observations: {h1_n}")
print(f"H2 observations: {h2_n}")
print(f"H3 observations: {h3_n}")

print("\nStudy periods:")
print(f"H1: {h1_start}–{h1_end}")
print(f"H2: {h2_start}–{h2_end}")
print(f"H3: {h3_start}–{h3_end}")

print("\nCountry counts:")
print(f"H1: {h1_sample['country_code'].nunique()}")
print(f"H2: {h2_sample['country_code'].nunique()}")
print(f"H3: {h3_sample['country_code'].nunique()}")

print("\nFinal checks:")
print("✓ H1 retains its valid analytical observations.")
print("✓ H2 uses observations with non-missing Gini.")
print("✓ H3 uses observations with non-missing Gini.")
print("✓ H3 interaction uses the centred Gini specification.")
print("✓ Analytical period is 2000–2022.")
print("✓ Country and year fixed effects are included.")
print("✓ Country-clustered standard errors are used.")

print("\nNo sample-size harmonisation is required unless the dissertation "
      "methodology explicitly specifies a common listwise-deleted sample.")

### 3.13 Final Hypothesis-Test Results

In [ ]:
# ============================================================
# 3.13 FINAL HYPOTHESIS-TEST RESULTS
# ============================================================

print("FINAL HYPOTHESIS-TEST RESULTS")
print("=" * 70)


def cluster_reference_pvalue(model, parameter, groups):
    """Supplementary two-sided t p-value using clusters - 1 degrees of freedom."""
    t_stat = model.params[parameter] / model.bse[parameter]
    df = int(pd.Series(groups).nunique() - 1)

    if df <= 0:
        return np.nan

    return 2 * stats.t.sf(abs(t_stat), df=df)


final_results = pd.DataFrame({
    "Hypothesis": ["H1", "H2", "H3"],
    "Dependent variable": [
        "GDP growth",
        "Gini coefficient",
        "GDP growth"
    ],
    "Key predictor": [
        "Tax reduction",
        "Tax reduction",
        "Tax reduction × Gini"
    ],
    "N": [
        int(h1_model.nobs),
        int(h2_model.nobs),
        int(h3_centered_model.nobs)
    ],
    "Clusters": [
        h1_sample["country_code"].nunique(),
        h2_sample["country_code"].nunique(),
        h3_sample["country_code"].nunique()
    ],
    "Coefficient": [
        h1_model.params["tax_reduction_clean_pp"],
        h2_model.params["tax_reduction_clean_pp"],
        h3_centered_model.params["tax_gini_interaction_centered"]
    ],
    "SE": [
        h1_model.bse["tax_reduction_clean_pp"],
        h2_model.bse["tax_reduction_clean_pp"],
        h3_centered_model.bse["tax_gini_interaction_centered"]
    ],
    "Asymptotic p-value": [
        h1_model.pvalues["tax_reduction_clean_pp"],
        h2_model.pvalues["tax_reduction_clean_pp"],
        h3_centered_model.pvalues[
            "tax_gini_interaction_centered"
        ]
    ],
    "Cluster-reference p-value": [
        cluster_reference_pvalue(
            h1_model,
            "tax_reduction_clean_pp",
            h1_sample["country_code"]
        ),
        cluster_reference_pvalue(
            h2_model,
            "tax_reduction_clean_pp",
            h2_sample["country_code"]
        ),
        cluster_reference_pvalue(
            h3_centered_model,
            "tax_gini_interaction_centered",
            h3_sample["country_code"]
        )
    ]
})

final_results[
    [
        "Coefficient",
        "SE",
        "Asymptotic p-value",
        "Cluster-reference p-value"
    ]
] = final_results[
    [
        "Coefficient",
        "SE",
        "Asymptotic p-value",
        "Cluster-reference p-value"
    ]
].round(4)

display(final_results)


### 3.14 Final Robustness Analysis Summary

In [ ]:
# ============================================================
# 3.14 FINAL ROBUSTNESS ANALYSIS SUMMARY
# ============================================================

print("ROBUSTNESS ANALYSIS SUMMARY")
print("=" * 70)

robustness_results = pd.DataFrame({
    "Hypothesis": [
        "H1",
        "H1",
        "H1",
        "H2",
        "H2",
        "H2",
        "H3",
        "H3"
    ],
    "Specification": [
        "Primary continuous measure (>= 0.01 pp)",
        "Binary tax-reduction indicator",
        "Continuous measure with >= 0.10 pp threshold",
        "Binary tax-reduction indicator",
        "PanelOLS two-way fixed effects",
        "Primary continuous measure",
        "Binary indicator × centred Gini",
        "Continuous × centred Gini"
    ],
    "Key coefficient": [
        h1_model.params["tax_reduction_clean_pp"],
        h1_binary_model.params["tax_reduction_indicator"],
        h1_threshold_model.params["tax_reduction_0_10_pp"],
        h2_indicator_model.params["tax_reduction_indicator"],
        h2_panel_model.params["tax_reduction_clean_pp"],
        h2_model.params["tax_reduction_clean_pp"],
        h3_indicator_model.params["indicator_gini_interaction"],
        h3_centered_model.params["tax_gini_interaction_centered"]
    ],
    "p-value": [
        h1_model.pvalues["tax_reduction_clean_pp"],
        h1_binary_model.pvalues["tax_reduction_indicator"],
        h1_threshold_model.pvalues["tax_reduction_0_10_pp"],
        h2_indicator_model.pvalues["tax_reduction_indicator"],
        h2_panel_model.pvalues["tax_reduction_clean_pp"],
        h2_model.pvalues["tax_reduction_clean_pp"],
        h3_indicator_model.pvalues["indicator_gini_interaction"],
        h3_centered_model.pvalues["tax_gini_interaction_centered"]
    ]
})

robustness_results[
    ["Key coefficient", "p-value"]
] = robustness_results[
    ["Key coefficient", "p-value"]
].round(4)

display(robustness_results)
